In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

2025-07-11 19:26:18.123482: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752261978.387921      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752261978.464110      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


/kaggle/input/pred_gesture_model/tensorflow2/default/1/pred_gesture_imu_lstm_model.keras
/kaggle/input/phase-prediction-model/tensorflow2/default/1/pred_phase_imu_lstm_model.keras
/kaggle/input/cmi-detect-behavior-with-sensor-data/train_demographics.csv
/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv
/kaggle/input/cmi-detect-behavior-with-sensor-data/train.csv
/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv
/kaggle/input/cmi-detect-behavior-with-sensor-data/kaggle_evaluation/cmi_inference_server.py
/kaggle/input/cmi-detect-behavior-with-sensor-data/kaggle_evaluation/cmi_gateway.py
/kaggle/input/cmi-detect-behavior-with-sensor-data/kaggle_evaluation/__init__.py
/kaggle/input/cmi-detect-behavior-with-sensor-data/kaggle_evaluation/core/templates.py
/kaggle/input/cmi-detect-behavior-with-sensor-data/kaggle_evaluation/core/base_gateway.py
/kaggle/input/cmi-detect-behavior-with-sensor-data/kaggle_evaluation/core/relay.py
/kaggle/input/cmi-detect-behavior-

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

In [3]:
train_data_df = pd.read_csv("/kaggle/input/cmi-detect-behavior-with-sensor-data/train.csv")
train_demo_df = pd.read_csv("/kaggle/input/cmi-detect-behavior-with-sensor-data/train_demographics.csv")
test_data_df = pd.read_csv("/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv")
test_demo_df = pd.read_csv("/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv")

# Preprocessing code

In [4]:
from scipy.spatial.transform import Rotation as R
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder

def add_imu_tabular_features(df):
    df = df.copy()

    # --- Orientation features ---
    rot_cols = ['rot_x', 'rot_y', 'rot_z', 'rot_w']
    quats = df[rot_cols].astype(float).values

    # Impute missing quaternions with identity quaternion [0,0,0,1]
    invalid_mask = ~np.isfinite(quats).all(axis=1)
    quats[invalid_mask] = [0, 0, 0, 1]

    norms = np.linalg.norm(quats, axis=1)
    zero_norm_mask = norms < 1e-6
    quats[zero_norm_mask] = [0, 0, 0, 1]

    # Normalize quaternions
    norms = np.linalg.norm(quats, axis=1, keepdims=True)
    quats_clean = quats / (norms + 1e-8)

    # Convert to Euler angles
    rotations = R.from_quat(quats_clean)
    eulers = rotations.as_euler('xyz', degrees=False)
    df['roll'] = eulers[:, 0]
    df['pitch'] = eulers[:, 1]
    df['yaw'] = eulers[:, 2]

    # Quaternion magnitude (excluding w)
    df['rot_magnitude'] = np.linalg.norm(quats[:, :3], axis=1)

    # Re-add normalized quaternion components
    df['rot_x_norm'] = quats_clean[:, 0]
    df['rot_y_norm'] = quats_clean[:, 1]
    df['rot_z_norm'] = quats_clean[:, 2]
    df['rot_w_norm'] = quats_clean[:, 3]

    # --- Acceleration features ---
    acc = df[['acc_x', 'acc_y', 'acc_z']].astype(float).values
    acc_x, acc_y, acc_z = acc[:, 0], acc[:, 1], acc[:, 2]

    df['acc_mag'] = np.linalg.norm(acc, axis=1)
    df['acc_xy_angle'] = np.arctan2(acc_y, acc_x)
    df['acc_balance_score'] = np.abs(acc_x - acc_y)
    df['acc_z_ratio'] = acc_z / (df['acc_mag'] + 1e-6)

    # Re-add raw acc
    df['acc_x'] = acc_x
    df['acc_y'] = acc_y
    df['acc_z'] = acc_z

    # --- Anthropometric features ---
    df['height_cm'] = df['height_cm'].astype(float)
    df['shoulder_to_wrist_cm'] = df['shoulder_to_wrist_cm'].astype(float)
    df['elbow_to_wrist_cm'] = df['elbow_to_wrist_cm'].astype(float)

    df['reach_total'] = df['shoulder_to_wrist_cm'] + df['elbow_to_wrist_cm']
    df['reach_ratio'] = df['reach_total'] / (df['height_cm'] + 1e-6)
    df['arm_ratio'] = df['elbow_to_wrist_cm'] / (df['shoulder_to_wrist_cm'] + 1e-6)

    df['age'] = df['age'].astype(float)

    return df

def imu_sequence_pipeline(data_df, demo_df, max_seq_len=700, mode='train', train_means=None):
    assert mode in ['train', 'test'], "mode must be either 'train' or 'test'"

    # --- Remove tof and temp columns ---
    tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(64)]
    temp_columns = [f'thm_{i}' for i in range(1, 6)]
    tof_temp_columns = tof_columns + temp_columns
    imu_only_data = data_df.drop(columns=tof_temp_columns, errors='ignore')

    # --- Merge with demographics ---
    imu_merged = pd.merge(imu_only_data, demo_df, on='subject', how='inner')

    # --- Feature engineering ---
    imu_merged = add_imu_tabular_features(imu_merged)

    # --- Temporal IMU columns ---
    temp_imu_cols = ['roll', 'pitch', 'yaw',
                 'rot_magnitude', 'rot_x_norm', 'rot_y_norm', 'rot_z_norm', 'rot_w_norm',
                 'acc_mag', 'acc_xy_angle', 'acc_balance_score', 'acc_z_ratio',
                 'acc_x', 'acc_y', 'acc_z',
                 'adult_child', 'sex', 'handedness',
                 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm',
                 'reach_ratio', 'arm_ratio']

    if mode == 'train':
        train_means = imu_merged[temp_imu_cols].mean()
        imu_merged[temp_imu_cols] = imu_merged[temp_imu_cols].fillna(train_means)
    else: 
        if train_means is not None:
            imu_merged[temp_imu_cols] = imu_merged[temp_imu_cols].fillna(train_means)
        else:
            raise ValueError("Must provide train_means during test mode")

    # --- Sort and group ---
    imu_merged_sorted = imu_merged.sort_values(['sequence_id', 'sequence_counter'])
    grouped = imu_merged_sorted.groupby('sequence_id')[temp_imu_cols]

    # Extract list of sequences
    X_seqs = grouped.apply(lambda x: x.values).tolist()

    # Extract sequence IDs in the same order
    sequence_ids = grouped.groups.keys()
    sequence_ids = list(sequence_ids)

    X_padded = pad_sequences(X_seqs, maxlen=max_seq_len, dtype='float32', padding='post', truncating='pre', value=0.0)

    if mode == 'test':
        if len(X_padded.shape) == 2:
            X_padded = np.expand_dims(X_padded, axis=0)
        return X_padded, sequence_ids

    # --- Only for training: create labels for gesture phase ---
    imu_merged_sorted['is_gesture'] = (imu_merged_sorted['phase'] == 'Gesture').astype(int)
    y_seqs = imu_merged_sorted.groupby('sequence_id')['is_gesture'].apply(lambda x: x.values).tolist()
    y_phase_padded = pad_sequences(y_seqs, maxlen=max_seq_len, dtype='int32', padding='post', truncating='pre', value=0)
    y_phase_padded = y_phase_padded[..., np.newaxis]

    # --- Only for training: create labels for gesture class ---
    gesture_series = imu_merged_sorted['gesture']

    # numerically encode class
    label_encoder = LabelEncoder()
    gesture_ids = label_encoder.fit_transform(gesture_series)

    imu_merged_sorted['gesture_id'] = gesture_ids
    # Group by sequence
    y_gesture_seqs = imu_merged_sorted.groupby('sequence_id')['gesture_id'].apply(lambda x: x.values).tolist()

    # Pad with number of classes
    y_gesture_padded = pad_sequences(y_gesture_seqs, maxlen=max_seq_len, dtype='int32', padding='post',truncating='pre', value=18)
    # y_gesture_padded = y_gesture_padded[..., np.newaxis]

    return X_padded, y_phase_padded, y_gesture_padded, label_encoder, sequence_ids, train_means

In [5]:
X_imu_phase_train, y_imu_phase_train, y_imu_gesture_train, label_encoder, sequence_ids, train_means = imu_sequence_pipeline(
    max_seq_len=700,
    data_df=train_data_df,
    demo_df=train_demo_df,
    mode='train'
)
print(X_imu_phase_train.shape)
print(y_imu_phase_train.shape)

(8151, 700, 23)
(8151, 700, 1)


In [6]:
# append is_gesture to training data
X_imu_gesture_train = np.concatenate([X_imu_phase_train, y_imu_phase_train], axis=-1)
print("training X shape:", X_imu_gesture_train.shape)
print("training y gesture shape:",y_imu_gesture_train.shape)
print(np.unique(y_imu_gesture_train))

training X shape: (8151, 700, 24)
training y gesture shape: (8151, 700)
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]


# Load Models

In [7]:
print("loading models...")
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False, reduction='none')

def masked_loss(y_true, y_pred):
    # y_true = tf.squeeze(y_true, axis=-1)  # if needed
    loss = loss_fn(y_true, y_pred)
    mask = tf.cast(tf.not_equal(y_true, padding_class), tf.float32)
    loss = loss * mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

padding_class = 18

def masked_accuracy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    # y_true = tf.squeeze(y_true, axis=-1)
    y_pred_labels = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
    mask = tf.not_equal(y_true, padding_class)
    correct = tf.equal(y_true, y_pred_labels)
    correct = tf.logical_and(correct, mask)
    correct = tf.cast(correct, tf.float32)
    mask = tf.cast(mask, tf.float32)
    return tf.reduce_sum(correct) / tf.reduce_sum(mask)

class StopTrainingOnAccuracy(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs.get('accuracy') is not None and logs.get('accuracy') >= 0.6 and logs.get('val_accuracy') is not None and logs.get('val_accuracy') >= 0.6:
            print("\nReached 60% validation accuracy, stopping training.")
            self.model.stop_training = True

loading models...


In [8]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Masking, TimeDistributed

tf.random.set_seed(0)   # always seed your experiments
Init = tf.keras.initializers.RandomNormal(seed=0)

class StopTrainingOnAccuracy_1(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs.get('accuracy') is not None and logs.get('accuracy') >= 0.99 and logs.get('val_accuracy') is not None and logs.get('val_accuracy') >= 0.996:
            print("\nReached 99.6% validation accuracy, stopping training.")
            self.model.stop_training = True

input_dim = X_imu_phase_train.shape[2]
max_seq_len = X_imu_phase_train.shape[1]

inputs = Input(shape=(max_seq_len, input_dim))

# mask padding values
x = Masking(mask_value=0.0)(inputs)

# One or more BiLSTM layers
x = tf.keras.layers.Bidirectional(LSTM(128, return_sequences=True))(x)
x = tf.keras.layers.Dropout(0.3)(x)

x = tf.keras.layers.Bidirectional(LSTM(80, return_sequences=True))(x)
x = tf.keras.layers.Dropout(0.3)(x)

# Optional dense layer
x = tf.keras.layers.Dense(40, activation='swish')(x)
x = tf.keras.layers.Dropout(0.3)(x)

# Output
outputs = tf.keras.layers.Dense(1, activation='softmax')(x)

pred_phase_imu_model = Model(inputs, outputs)

pred_phase_imu_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# pred_phase_imu_model.fit(X_imu_phase_train, y_imu_phase_train, batch_size=1000, epochs=100, callbacks=[StopTrainingOnAccuracy_1()], validation_split=0.1)

2025-07-11 19:27:13.976414: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [9]:
Init = tf.keras.initializers.RandomNormal(seed=0)

class StopTrainingOnAccuracy(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs.get('accuracy') is not None and logs.get('accuracy') >= 0.6 and logs.get('val_accuracy') is not None and logs.get('val_accuracy') >= 0.6:
            print("\nReached 60% validation accuracy, stopping training.")
            self.model.stop_training = True

input_dim = X_imu_gesture_train.shape[2]
max_seq_len = X_imu_gesture_train.shape[1]

inputs = Input(shape=(max_seq_len, input_dim))

# mask padding values
x = Masking(mask_value=0.0)(inputs)

# One or more BiLSTM layers
x = tf.keras.layers.Bidirectional(LSTM(128, return_sequences=True))(x)
x = tf.keras.layers.Dropout(0.3)(x)

x = tf.keras.layers.Bidirectional(LSTM(80, return_sequences=True))(x)
x = tf.keras.layers.Dropout(0.3)(x)

# Optional dense layer
x = tf.keras.layers.Dense(40, activation='swish')(x)
x = tf.keras.layers.Dropout(0.3)(x)

# Output
outputs = tf.keras.layers.Dense(19, activation='softmax')(x)

# learning rate scheduler
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    0.001,
    decay_steps=400,
    decay_rate=0.96,
    staircase=True
)

# Nadam optimizer
imu_optimizer = tf.keras.optimizers.Nadam(learning_rate=lr_schedule)

# model
pred_gesture_imu_model = Model(inputs, outputs)

pred_gesture_imu_model.compile(
    optimizer=imu_optimizer,
    loss=masked_loss,
    metrics=[masked_accuracy]
)

# pred_gesture_imu_model.fit(
#     X_imu_gesture_train, y_imu_gesture_train,
#     validation_split=0.1,
#     sample_weight=sample_weight,
#     batch_size=40,
#     epochs=100,
#     callbacks=[StopTrainingOnAccuracy()]
# )

In [10]:
from tensorflow.keras.models import load_model
pred_phase_imu_model = load_model('/kaggle/input/phase-prediction-model/tensorflow2/default/1/pred_phase_imu_lstm_model.keras')

pred_gesture_imu_model = load_model('/kaggle/input/pred_gesture_model/tensorflow2/default/1/pred_gesture_imu_lstm_model.keras', custom_objects={
        "masked_loss": masked_loss,
        "masked_accuracy": masked_accuracy,
        "StopTrainingOnAccuracy": StopTrainingOnAccuracy
    })
print("loaded models!")
print("making predictions...")

loaded models!
making predictions...


In [11]:
import os

import pandas as pd
import polars as pl

import sys
sys.path.append('/kaggle/input/cmi-detect-behavior-with-sensor-data')

import kaggle_evaluation.cmi_inference_server


In [12]:
import polars as pl
import numpy as np
from scipy.stats import mode
def predict(sequence: pl.DataFrame, demographics: pl.DataFrame) -> str:
    # Replace this function with your inference code.
    # You can return either a Pandas or Polars dataframe, though Polars is recommended.
    # Each prediction (except the very first) must be returned within 30 minutes of the batch features being provided.
    # extract test data
    data_df=sequence.to_pandas()
    demo_df=demographics.to_pandas()
    
    X_imu_phase_test, test_sequence_ids = imu_sequence_pipeline(
    max_seq_len=700,
    data_df=data_df,
    demo_df=demo_df,
    mode='test', 
    train_means = train_means)
    # predict gesture phase with model
    test_y_imu_phase_pred = pred_phase_imu_model.predict(X_imu_phase_test)
    test_y_imu_phase_binary = (test_y_imu_phase_pred > 0.5).astype(int)

    X_imu_gesture_test = np.concatenate([X_imu_phase_test, test_y_imu_phase_binary], axis=-1)
    # predict gesture
    test_y_imu_gesture_pred = pred_gesture_imu_model.predict(X_imu_gesture_test)
    
    # postprocess probabilities
    phase_mask = test_y_imu_phase_binary.squeeze(-1).astype(bool) 

    # Step 2: get max class for each
    adjusted_preds = np.argmax(test_y_imu_gesture_pred, axis=-1)  # shape (8101, 700)

    # get majority class
    valid_timesteps_preds = adjusted_preds[phase_mask]  # shape: (num_valid_timesteps, num_classes)
    if len(valid_timesteps_preds) == 0:
        values, counts = np.unique(adjusted_preds, return_counts=True)
        sorted_indices = np.argsort(-counts)
        sorted_values = values[sorted_indices]
        if len(sorted_values) >= 2:
            majority_class = int(sorted_values[1])
        elif len(sorted_values) == 1:
            majority_class = int(sorted_values[0])
        else:
            majority_class = 0  # fallback default if empty
    else:
        majority_class = int(mode(valid_timesteps_preds, keepdims=False).mode.item())
    prediction = label_encoder.inverse_transform([majority_class])[0]
    return prediction

In [13]:
inference_server = kaggle_evaluation.cmi_inference_server.CMIInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        data_paths=(
            '/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv',
            '/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv',
        )
    )

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 239ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step
